# Module 8C: DQ Dashboard - Streamlit in Snowflake

## Learning Objectives
- Build and deploy a **Streamlit in Snowflake (SiS)** DQ monitoring app
- Provide self-service DQ visibility to business users

> **Business Value:** Business users get a polished, interactive dashboard inside Snowflake. No Power BI license, no VPN, no external hosting -- just Snowflake credentials and a browser.

---
> **Role:** `CORP_DQ_ADMIN` | **Time:** ~45 minutes | **Variant:** Streamlit in Snowflake

> **What this does:** Sets your session context to the lab role, database, and warehouse.


In [ ]:
USE ROLE CORP_DQ_ADMIN;
USE DATABASE CORP_DWH;
USE WAREHOUSE COMPUTE_WH;

---
## Shared Setup: Create DQ Reporting Views

> **Note:** These views are identical across all Module 8 variants (8A/8B/8C). If you already ran another variant, these views already exist -- running them again is safe (`CREATE OR REPLACE`).

> **Business Value:** BI tools cannot call table functions directly. Views provide a stable, queryable interface.

In [ ]:
CREATE OR REPLACE VIEW CORP_DWH.DQ.V_DQ_RESULTS_FLAT AS
SELECT
    r.REF_ENTITY_NAME AS TABLE_NAME, r.METRIC_NAME,
    r.ARGUMENT_NAMES AS COLUMN_CHECKED, r.VALUE AS METRIC_VALUE,
    r.EXPECTATION_NAME, r.EXPECTATION_RESULT, r.MEASUREMENT_TIME,
    COALESCE(c.SEVERITY, 'MEDIUM') AS SEVERITY,
    COALESCE(c.OWNER, 'Unassigned') AS RULE_OWNER,
    COALESCE(c.RULE_TYPE, 'SYSTEM') AS RULE_TYPE,
    CASE WHEN r.EXPECTATION_RESULT = 'MET' THEN 'PASS'
         WHEN r.EXPECTATION_RESULT = 'NOT_MET' THEN 'FAIL'
         ELSE 'NO_EXPECTATION' END AS STATUS
FROM TABLE(SNOWFLAKE.LOCAL.DATA_QUALITY_MONITORING_RESULTS(
    REF_ENTITY_NAME => 'CORP_DWH.GOLD.DIM_CUSTOMER', REF_ENTITY_DOMAIN => 'TABLE'
)) r
LEFT JOIN CORP_DWH.DQ.RULES_CATALOG c
    ON UPPER(r.METRIC_NAME) LIKE '%' || REPLACE(UPPER(c.RULE_NAME), ' ', '_') || '%';

> **What this does:** Creates V_DQ_SCORECARD view that calculates per-table health scores from the latest DQ expectation results.


In [ ]:
CREATE OR REPLACE VIEW CORP_DWH.DQ.V_DQ_SCORECARD AS
WITH latest AS (
    SELECT 'CORP_DWH.GOLD.DIM_CUSTOMER' AS TABLE_NAME,
        METRIC_NAME, ARGUMENT_NAMES, VALUE, EXPECTATION_NAME, EXPECTATION_RESULT, MEASUREMENT_TIME,
        ROW_NUMBER() OVER (PARTITION BY METRIC_NAME, ARGUMENT_NAMES ORDER BY MEASUREMENT_TIME DESC) AS RN
    FROM TABLE(SNOWFLAKE.LOCAL.DATA_QUALITY_MONITORING_RESULTS(
        REF_ENTITY_NAME => 'CORP_DWH.GOLD.DIM_CUSTOMER', REF_ENTITY_DOMAIN => 'TABLE'))
    WHERE EXPECTATION_NAME IS NOT NULL
)
SELECT TABLE_NAME,
    COUNT(*) AS TOTAL_EXPECTATIONS,
    COUNT(CASE WHEN EXPECTATION_RESULT = 'MET' THEN 1 END) AS PASSED,
    COUNT(CASE WHEN EXPECTATION_RESULT = 'NOT_MET' THEN 1 END) AS FAILED,
    ROUND(100.0 * COUNT(CASE WHEN EXPECTATION_RESULT = 'MET' THEN 1 END) / NULLIF(COUNT(*), 0), 1) AS HEALTH_SCORE_PCT,
    MAX(MEASUREMENT_TIME) AS LAST_EVALUATED
FROM latest WHERE RN = 1 GROUP BY TABLE_NAME;

> **What this does:** Creates V_DQ_TREND view that provides hourly time-series data for charting quality metrics over time.


In [ ]:
CREATE OR REPLACE VIEW CORP_DWH.DQ.V_DQ_TREND AS
SELECT DATE_TRUNC('HOUR', MEASUREMENT_TIME) AS MEASUREMENT_HOUR,
    METRIC_NAME, VALUE AS METRIC_VALUE, EXPECTATION_RESULT, MEASUREMENT_TIME
FROM TABLE(SNOWFLAKE.LOCAL.DATA_QUALITY_MONITORING_RESULTS(
    REF_ENTITY_NAME => 'CORP_DWH.GOLD.DIM_CUSTOMER', REF_ENTITY_DOMAIN => 'TABLE'))
WHERE EXPECTATION_NAME IS NOT NULL ORDER BY MEASUREMENT_TIME DESC;

> **What this does:** Creates V_DQ_EXECUTIVE_SUMMARY view that aggregates all checks into a single overall health percentage.


In [ ]:
CREATE OR REPLACE VIEW CORP_DWH.DQ.V_DQ_EXECUTIVE_SUMMARY AS
SELECT 'CORP_DWH' AS DATA_ESTATE, COUNT(*) AS TOTAL_CHECKS,
    COUNT(CASE WHEN EXPECTATION_RESULT = 'MET' THEN 1 END) AS CHECKS_PASSING,
    COUNT(CASE WHEN EXPECTATION_RESULT = 'NOT_MET' THEN 1 END) AS CHECKS_FAILING,
    ROUND(100.0 * COUNT(CASE WHEN EXPECTATION_RESULT = 'MET' THEN 1 END) / NULLIF(COUNT(*), 0), 1) AS OVERALL_HEALTH_PCT,
    CURRENT_TIMESTAMP() AS AS_OF
FROM TABLE(SNOWFLAKE.LOCAL.DATA_QUALITY_MONITORING_RESULTS(
    REF_ENTITY_NAME => 'CORP_DWH.GOLD.DIM_CUSTOMER', REF_ENTITY_DOMAIN => 'TABLE'))
WHERE EXPECTATION_NAME IS NOT NULL;

---
## Create the App Schema and Stage

> **What this does:** Creates the APPS schema and a stage to host the Streamlit app files.


In [ ]:
CREATE SCHEMA IF NOT EXISTS CORP_DWH.APPS COMMENT = 'Streamlit applications';
CREATE OR REPLACE STAGE CORP_DWH.APPS.DQ_DASHBOARD_STAGE DIRECTORY = (ENABLE = TRUE);

---
## The Streamlit App Code

Below is the complete app. Review it, then we deploy it to Snowflake.

In [ ]:
# Preview the Streamlit app code
app_code = open('/tmp/dq_dashboard_app.py', 'w')
app_code.write('''import streamlit as st
from snowflake.snowpark.context import get_active_session

st.set_page_config(page_title="DQ Monitor", page_icon="shield", layout="wide")
session = get_active_session()

st.title("Data Quality Monitoring Dashboard")
st.caption("Real-time quality metrics for CORP_DWH")

exec_df = session.sql("SELECT * FROM CORP_DWH.DQ.V_DQ_EXECUTIVE_SUMMARY").to_pandas()

if not exec_df.empty:
    col1, col2, col3, col4 = st.columns(4)
    health = float(exec_df["OVERALL_HEALTH_PCT"].iloc[0] or 0)
    total = int(exec_df["TOTAL_CHECKS"].iloc[0] or 0)
    passing = int(exec_df["CHECKS_PASSING"].iloc[0] or 0)
    failing = int(exec_df["CHECKS_FAILING"].iloc[0] or 0)
    col1.metric("Overall Health", f"{health:.0f}%")
    col2.metric("Total Checks", total)
    col3.metric("Passing", passing)
    col4.metric("Failing", failing)
else:
    st.warning("No DQ results available yet.")

st.divider()
tab1, tab2, tab3 = st.tabs(["Scorecard", "Failing Rules", "Rules Catalog"])

with tab1:
    st.subheader("Health by Table")
    scorecard = session.sql("SELECT * FROM CORP_DWH.DQ.V_DQ_SCORECARD").to_pandas()
    if not scorecard.empty:
        for _, row in scorecard.iterrows():
            h = float(row["HEALTH_SCORE_PCT"] or 0)
            st.progress(h / 100, text=f"{row['TABLE_NAME']} -- {h:.0f}%")
    else:
        st.info("No scorecard data.")

with tab2:
    st.subheader("Top Failing Rules")
    fails = session.sql(
        "SELECT METRIC_NAME AS RULE, COLUMN_CHECKED, METRIC_VALUE AS VIOLATIONS, SEVERITY "
        "FROM CORP_DWH.DQ.V_DQ_RESULTS_FLAT WHERE STATUS = 'FAIL' AND METRIC_VALUE > 0 "
        "ORDER BY METRIC_VALUE DESC LIMIT 10"
    ).to_pandas()
    if not fails.empty:
        st.dataframe(fails, use_container_width=True, hide_index=True)
    else:
        st.success("No failures detected!")

with tab3:
    st.subheader("Active Rules Catalog")
    catalog = session.sql(
        "SELECT RULE_NAME, RULE_TYPE, TARGET_TABLE, SEVERITY, OWNER, "
        "CASE WHEN DMF_NAME IS NOT NULL THEN 'Provisioned' ELSE 'Pending' END AS STATUS "
        "FROM CORP_DWH.DQ.RULES_CATALOG WHERE IS_ACTIVE = TRUE ORDER BY SEVERITY DESC"
    ).to_pandas()
    if not catalog.empty:
        st.dataframe(catalog, use_container_width=True, hide_index=True)
''')
app_code.close()
print('App code written. Full listing:')
print('=' * 60)
print(open('/tmp/dq_dashboard_app.py').read())
print('=' * 60)

---
## Deploy the App

> **What this does:** Uploads the Streamlit app code to the stage and creates the Streamlit object in Snowflake for deployment.


In [ ]:
from snowflake.snowpark.context import get_active_session
import tempfile, os
session = get_active_session()

# Write app code to temp file
app_code = '''import streamlit as st
from snowflake.snowpark.context import get_active_session

st.set_page_config(page_title="DQ Monitor", page_icon="shield", layout="wide")
session = get_active_session()

st.title("Data Quality Monitoring Dashboard")
st.caption("Real-time quality metrics for CORP_DWH")

exec_df = session.sql("SELECT * FROM CORP_DWH.DQ.V_DQ_EXECUTIVE_SUMMARY").to_pandas()

if not exec_df.empty:
    col1, col2, col3, col4 = st.columns(4)
    health = float(exec_df["OVERALL_HEALTH_PCT"].iloc[0] or 0)
    total = int(exec_df["TOTAL_CHECKS"].iloc[0] or 0)
    passing = int(exec_df["CHECKS_PASSING"].iloc[0] or 0)
    failing = int(exec_df["CHECKS_FAILING"].iloc[0] or 0)
    col1.metric("Overall Health", f"{health:.0f}%")
    col2.metric("Total Checks", total)
    col3.metric("Passing", passing)
    col4.metric("Failing", failing)
else:
    st.warning("No DQ results available yet.")

st.divider()
tab1, tab2, tab3 = st.tabs(["Scorecard", "Failing Rules", "Rules Catalog"])

with tab1:
    st.subheader("Health by Table")
    scorecard = session.sql("SELECT * FROM CORP_DWH.DQ.V_DQ_SCORECARD").to_pandas()
    if not scorecard.empty:
        for _, row in scorecard.iterrows():
            h = float(row["HEALTH_SCORE_PCT"] or 0)
            st.progress(h / 100, text=f"{row['TABLE_NAME']} -- {h:.0f}%")
    else:
        st.info("No scorecard data.")

with tab2:
    st.subheader("Top Failing Rules")
    fails = session.sql(
        "SELECT METRIC_NAME AS RULE, COLUMN_CHECKED, METRIC_VALUE AS VIOLATIONS, SEVERITY "
        "FROM CORP_DWH.DQ.V_DQ_RESULTS_FLAT WHERE STATUS = 'FAIL' AND METRIC_VALUE > 0 "
        "ORDER BY METRIC_VALUE DESC LIMIT 10"
    ).to_pandas()
    if not fails.empty:
        st.dataframe(fails, use_container_width=True, hide_index=True)
    else:
        st.success("No failures detected!")

with tab3:
    st.subheader("Active Rules Catalog")
    catalog = session.sql(
        "SELECT RULE_NAME, RULE_TYPE, TARGET_TABLE, SEVERITY, OWNER, "
        "CASE WHEN DMF_NAME IS NOT NULL THEN 'Provisioned' ELSE 'Pending' END AS STATUS "
        "FROM CORP_DWH.DQ.RULES_CATALOG WHERE IS_ACTIVE = TRUE ORDER BY SEVERITY DESC"
    ).to_pandas()
    if not catalog.empty:
        st.dataframe(catalog, use_container_width=True, hide_index=True)
'''

tmp = tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False)
tmp.write(app_code)
tmp.close()

try:
    session.file.put(tmp.name, "@CORP_DWH.APPS.DQ_DASHBOARD_STAGE/", auto_compress=False, overwrite=True)
    # Rename uploaded file to app.py
    uploaded_name = os.path.basename(tmp.name)
    print(f"[OK] Uploaded {uploaded_name} to stage")
    
    session.sql(f"""
        CREATE OR REPLACE STREAMLIT CORP_DWH.APPS.DQ_DASHBOARD
            ROOT_LOCATION = '@CORP_DWH.APPS.DQ_DASHBOARD_STAGE'
            MAIN_FILE = '{uploaded_name}'
            QUERY_WAREHOUSE = COMPUTE_WH
            COMMENT = 'Data Quality Monitoring Dashboard'
    """).collect()
    print("[OK] Streamlit app created: CORP_DWH.APPS.DQ_DASHBOARD")
    print("\nAccess: Snowflake UI > Projects > Streamlit > DQ_DASHBOARD")
except Exception as e:
    print(f"[INFO] {str(e)[:120]}")
    print("\nManual deploy: Go to Projects > Streamlit > + Streamlit App")
    print("Paste the app code shown above.")
finally:
    os.unlink(tmp.name)

---
## Grant Access to Business Users

> **What this does:** Grants the DQ steward role access to the deployed Streamlit dashboard.


In [ ]:
GRANT USAGE ON STREAMLIT CORP_DWH.APPS.DQ_DASHBOARD TO ROLE CORP_DQ_STEWARD;

---
## Comparison: When to Use Each Variant

| Variant | Best For | Pros | Cons |
|---------|----------|------|------|
| **8A: Native Dashboards** | Quick SQL monitoring | Zero code, built-in | Limited interactivity |
| **8B: Python Charts** | Data engineers | Rich customization | Not shareable standalone |
| **8C: Streamlit (SiS)** | Business self-service | Interactive, deployed, shareable | Requires app code |
| **Power BI** | Enterprise BI teams | Familiar, RLS, scheduling | External tool, license |

---
## Checkpoint

> **What this does:** Verifies your work so far. All checks should show [PASS].


In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
print("=" * 50)
print("CHECKPOINT: Dashboard Views + Streamlit App")
print("=" * 50)

passed = 0
total = 5

# Check 4 dashboard views
for v in ['V_DQ_RESULTS_FLAT', 'V_DQ_SCORECARD', 'V_DQ_TREND', 'V_DQ_EXECUTIVE_SUMMARY']:
    try:
        cnt = session.sql(f"SELECT COUNT(*) AS C FROM CORP_DWH.DQ.{v}").collect()[0]['C']
        print(f"  [PASS] {v} -- {cnt} rows")
        passed += 1
    except Exception as e:
        print(f"  [FAIL] {v}: {str(e)[:50]}")

# Check Streamlit app exists
try:
    result = session.sql("SHOW STREAMLIT LIKE 'DQ_DASHBOARD' IN SCHEMA CORP_DWH.APPS").collect()
    if len(result) > 0:
        print(f"  [PASS] Streamlit app DQ_DASHBOARD deployed")
        passed += 1
    else:
        print(f"  [WAIT] Streamlit app not yet deployed (run the deploy cell above)")
except Exception as e:
    print(f"  [WAIT] Streamlit app not yet deployed: {str(e)[:50]}")

print(f"\nResult: {passed}/{total} checks passed")
print("=" * 50)


---
**Next:** Proceed to `9_TEARDOWN` (optional cleanup).